In [1]:
import json
import optuna
import warnings
import pandas as pd
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

from utils.tuning import (
    tune_hyperparameters,
    get_hyperparams_getter_function_by_model_name,
    prepare_hyperparams,
)
from utils.config import load_config
from utils.shared import get_model_by_name

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

c:\Users\Zhenia\Desktop\ml_fbm_2\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\Zhenia\Desktop\ml_fbm_2\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\Zhenia\Desktop\ml_fbm_2\.venv\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the genc

In [2]:
config = load_config()

In [3]:
for model in config.models:
    for data_dim in config.sample_sizes:

        print(
            f"Start tuning {model} model for {data_dim}x{config.train_samples_num} data..."
        )

        estimator = get_model_by_name(model, random_state=config.random_seed, n_jobs=config.n_jobs, input_shape=(data_dim, 1))
        hyperparams_getter_func = get_hyperparams_getter_function_by_model_name(model)
        data = pd.read_csv(
            f"{config.data_dir}/{config.train_data_subdir}/fbm_{data_dim}x{config.train_samples_num}.csv"
        )

        if model == "RNN" or model == "CNN":
            data_train, data_val = train_test_split(data, test_size=config.test_split_ratio, random_state=config.random_seed)

            study = tune_hyperparameters(
                experiment_name=config.tunning_experiment_name,
                run_name=f"{model}_{data_dim}",
                estimator=estimator,
                hyperparams_getter_func=hyperparams_getter_func,
                scoring_func=root_mean_squared_error,
                direction="minimize",
                X_train=data_train.drop(columns=[config.target_column]),
                y_train=data_train[config.target_column],
                X_val=data_val.drop(columns=[config.target_column]),
                y_val=data_val[config.target_column],
                n_trials=config.trials[model],
                cpus_to_use=config.n_jobs,
            )
        else:
            study = tune_hyperparameters(
                experiment_name=config.tunning_experiment_name,
                run_name=f"{model}_{data_dim}",
                estimator=estimator,
                hyperparams_getter_func=hyperparams_getter_func,
                scoring_func=root_mean_squared_error,
                direction="minimize",
                X_train=data.drop(columns=[config.target_column]),
                y_train=data[config.target_column],
                cv_folds=config.cv_folds,
                n_trials=config.trials[model],
                cpus_to_use=config.n_jobs,
            )

        # save best hyperparameters
        with open(f"hyperparams/{model}_{data_dim}.json", "w") as f:
            json.dump(prepare_hyperparams(study.best_params, estimator), f, indent=4)

        print(f"Tunning is finished.")

Start tuning CNN model for 25x100000 data...


  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 1/24
 360/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 0.2902[W 2025-12-14 22:55:44,834] Trial 0 failed with parameters: {'n_filters': 126, 'n_conv_layers': 3, 'kernel_size': 4, 'dense_units': 171, 'dropout': 0.12809363026119602, 'learning_rate': 0.007076381607570283, 'batch_size': 64, 'epochs': 24} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\Zhenia\Desktop\ml_fbm_2\.venv\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  File "c:\Users\Zhenia\Desktop\ml_fbm_2\utils\tuning.py", line 58, in objective
    y_pred = estimator.fit(X_train, y_train).predict(X_val)
             ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "c:\Users\Zhenia\Desktop\ml_fbm_2\utils\shared.py", line 108, in fit
    return super().fit(X, y, **kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "c:\Users\Zhenia\Desktop\ml_fbm_2\.venv\Lib\site-packages\scikeras\wrappers.py", line 770, in fit
    self

KeyboardInterrupt: 